<a href="https://colab.research.google.com/github/JESUSJEREZ/TALLER_UNIR/blob/main/Proyecto_Structured_Streaming_y_Kafka.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Structured Streaming y Kafka

**Presentado Por:** Jesus Eduardo Jerez Rojas



### Punto de partida (final de la actividad 1): función `retrasoMedio`
***Para los vuelos que llegan con retraso positivo, calcular para cada aeropuerto de llegada el retraso medio.***

Recordatorio: *El código que calcule esto debería ir encapsulado en una función de Python que reciba como argumento un DataFrame y devuelva como resultado el DataFrame con el cálculo del retraso medio por aeropuerto, ordenado de mayor a menor retraso medio. La columna creada con el retraso medio debe llamarse `retraso_medio`.*

**Copia en la siguiente celda el código de tu función retrasoMedio que has completado en la actividad 1**. El DataFrame devuelto por la función debería tener solamente dos columnas: `dest` y `retraso_medio`.

In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget https://downloads.apache.org/spark/spark-3.2.4/spark-3.2.4-bin-hadoop2.7.tgz
!tar -xvf spark-3.2.4-bin-hadoop2.7.tgz
!pip install -q findspark

--2023-09-26 00:57:57--  https://downloads.apache.org/spark/spark-3.2.4/spark-3.2.4-bin-hadoop2.7.tgz
Resolving downloads.apache.org (downloads.apache.org)... 135.181.214.104, 88.99.95.219, 2a01:4f8:10a:201a::2, ...
Connecting to downloads.apache.org (downloads.apache.org)|135.181.214.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 272938638 (260M) [application/x-gzip]
Saving to: ‘spark-3.2.4-bin-hadoop2.7.tgz’

spark-3.2.4-bin-had 100%[===================>] 260.29M  12.8MB/s    in 22s     

2023-09-26 00:58:20 (11.6 MB/s) - ‘spark-3.2.4-bin-hadoop2.7.tgz’ saved [272938638/272938638]

spark-3.2.4-bin-hadoop2.7/
spark-3.2.4-bin-hadoop2.7/R/
spark-3.2.4-bin-hadoop2.7/R/lib/
spark-3.2.4-bin-hadoop2.7/R/lib/sparkr.zip
spark-3.2.4-bin-hadoop2.7/R/lib/SparkR/
spark-3.2.4-bin-hadoop2.7/R/lib/SparkR/html/
spark-3.2.4-bin-hadoop2.7/R/lib/SparkR/html/00Index.html
spark-3.2.4-bin-hadoop2.7/R/lib/SparkR/html/R.css
spark-3.2.4-bin-hadoop2.7/R/lib/SparkR/INDEX
spark-3.2

In [ ]:
pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.8/310.8 MB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.4.1-py2.py3-none-any.whl size=311285387 sha256=86d075eb58e259c0de9a9f7a8fb5c8e40d74d44ffca8d54dab62aa1f4d5510a6
  Stored in directory: /root/.cache/pip/wheels/0d/77/a3/ff2f74cc9ab41f8f594dabf0579c2a7c6de920d584206e0834
Successfully built pyspark


In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession \
.builder \
.appName("Streaming from Kafka") \
.config("spark.streaming.stopGracefullyOnShutdown", True) \
.config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0') \
.config("spark.sql.shuffle.partitions", 4) \
.master("local[*]") \
.getOrCreate()
spark

In [ ]:
path_file="flights_act1.csv"

flightsDF=spark.read.option("header","true").option("inferSchema", "true").csv(path_file)

In [ ]:
flightsDF.count()

162049

In [ ]:
flightsDF.printSchema()

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- dep_time: string (nullable = true)
 |-- dep_delay: string (nullable = true)
 |-- arr_time: string (nullable = true)
 |-- arr_delay: string (nullable = true)
 |-- carrier: string (nullable = true)
 |-- tailnum: string (nullable = true)
 |-- flight: integer (nullable = true)
 |-- origin: string (nullable = true)
 |-- dest: string (nullable = true)
 |-- air_time: string (nullable = true)
 |-- distance: integer (nullable = true)
 |-- hour: string (nullable = true)
 |-- minute: string (nullable = true)



In [ ]:
from pyspark.sql import functions as F
cuantos_NA = flightsDF\
                .where(F.col("dep_time") == "NA")\
                .count()
cuantos_NA

857

In [ ]:
columnas_limpiar = ["dep_time", "dep_delay", "arr_time", "arr_delay", "air_time", "hour", "minute"]

flightsLimpiado = flightsDF
for nombreColumna in columnas_limpiar:  # para cada columna, nos quedamos con las filas que no tienen NA en esa columna
    flightsLimpiado = flightsLimpiado.where(F.col(nombreColumna) != "NA")

flightsLimpiado.cache()

DataFrame[year: int, month: int, day: int, dep_time: string, dep_delay: string, arr_time: string, arr_delay: string, carrier: string, tailnum: string, flight: int, origin: string, dest: string, air_time: string, distance: int, hour: string, minute: string]

In [ ]:
flightsLimpiado.count()

160748

In [ ]:
from pyspark.sql.types import IntegerType, DoubleType

flightsConvertido = flightsLimpiado

for c in columnas_limpiar:
    # método que crea una columna o reemplaza una existente
    flightsConvertido = flightsConvertido.withColumn(c, F.col(c).cast(IntegerType()))

flightsConvertido = flightsConvertido.withColumn("arr_delay", F.col("arr_delay").cast(DoubleType()))
flightsConvertido.cache()

DataFrame[year: int, month: int, day: int, dep_time: int, dep_delay: int, arr_time: int, arr_delay: double, carrier: string, tailnum: string, flight: int, origin: string, dest: string, air_time: int, distance: int, hour: int, minute: int]

In [ ]:
flightsConvertido.printSchema()

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- dep_time: integer (nullable = true)
 |-- dep_delay: integer (nullable = true)
 |-- arr_time: integer (nullable = true)
 |-- arr_delay: double (nullable = true)
 |-- carrier: string (nullable = true)
 |-- tailnum: string (nullable = true)
 |-- flight: integer (nullable = true)
 |-- origin: string (nullable = true)
 |-- dest: string (nullable = true)
 |-- air_time: integer (nullable = true)
 |-- distance: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- minute: integer (nullable = true)



In [ ]:
aeropuertosOrigenDF =flightsConvertido.select("origin").distinct()
n_origen=aeropuertosOrigenDF.count()
rutasDistintasDF=flightsConvertido.select(["origin","dest"]).distinct()
n_rutas=rutasDistintasDF.count()
aeropuertosOrigenDF.cache()
rutasDistintasDF.cache()

DataFrame[origin: string, dest: string]

In [ ]:
assert(n_origen == 2)
assert(n_rutas == 115)
assert(aeropuertosOrigenDF.count() == n_origen)
assert(rutasDistintasDF.count() == n_rutas)

In [ ]:
def retrasoMedio(retrasoMedioDF):
    retrasoMedioDF=flightsConvertido.select("arr_delay","dest").filter(flightsConvertido.select["arr_delay"] >=0). withColumn("retraso_medio",F.groupby("dest").agg(F.avg("arr_delay"))).collect()(0)(0).orderby("retraso_medio",ascending=False)
    return retrasoMedioDF

In [ ]:
lista = retrasoMedio(flightsConvertido).take(3)
assert((lista[0].retraso_medio == 64.75) & (lista[0].dest == "BOI"))
assert((lista[1].retraso_medio == 46.8) & (lista[1].dest == "HDN"))
assert((round(lista[2].retraso_medio, 2) == 41.19) & (lista[2].dest == "SFO"))

TypeError: ignored

In [ ]:
from pyspark.sql.functions import col, when, avg, desc

def retrasoMedio(df):

    vuelos_con_retraso = df.filter(col("arr_delay") > 0)
    retrasoMedioDF = vuelos_con_retraso.groupBy("dest").agg({"arr_delay":"avg"}).withColumnRenamed("avg(arr_delay)","retraso_medio").orderBy(col("retraso_medio").desc())
    retrasoMedioDF = retrasoMedioDF.orderBy(desc("retraso_medio"))
    return retrasoMedioDF

    vuelos_con_retraso = df[df['arr_delay'] > 0]

In [ ]:
retrasoMedioDF = retrasoMedio(flightsConvertido)
retrasoMedioDF.limit(3).show()

+----+------------------+
|dest|     retraso_medio|
+----+------------------+
| BOI|             64.75|
| HDN|              46.8|
| SFO|41.193768844221104|
+----+------------------+



###**Inicio del Taller_2: Structured Streaming y Kafka**###

Utilizaremos Kafka para actualizar en tiempo real el resultado calculado en el apartado anterior.

Para simplificar, asumimos que los mensajes leídos de Kafka tiene solamente dos campos que son los únicos necesarios para llevar a cabo la operación anterior: dest y arr_delay. La idea será crear un Streaming DataFrame para leer de Kafka, y después invocar a nuestra función retrasoMedio pasándolo como argumento. Vamos a leer del topic `retrasos` por lo que debes indicar esta opción a continuación.

Se pide crear, en la variable `retrasosStreamingDF`, un Streaming DataFrame leyendo de Apache Kafka, configurando las siguientes opciones:
  * Usar la variable `readStream` (en lugar de `read` como solemos hacer) interna de la SparkSession `spark`
  * Indicar que el formato es `"kafka"` con `.format("kafka")`
  * Indicar cuáles son los brokers de Kafka de los que vamos a leer y el puerto al que queremos conectarnos para leer (9092 es el que usa Kafka por defecto), con `.option("kafka.bootstrap.servers", "<nombre_cluster>-w-0:9092,<nombre_cluster>-w-1:9092")`. De esa manera podremos leer el mensaje si el productor de Kafka lo envía a cualquiera de los dos brokers existentes, que son los nodos del cluster identificados como `<nombre_cluster>-w-0` y `<nombre_cluster>-w-1`
  * Indicar que queremos subscribirnos al topic `"retrasos"` con `.option("subscribe", "retrasos")`.
  * Finalmente ponemos `load()` para realizar la lectura.

In [ ]:
!pip install kafka-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.5/246.5 kB 3.3 MB/s eta 0:00:00


In [ ]:
!curl -sSOL https://downloads.apache.org/kafka/3.5.1/kafka_2.13-3.5.1.tgz
!tar -xvf kafka_2.13-3.5.1.tgz

kafka_2.13-3.5.1/
kafka_2.13-3.5.1/LICENSE
kafka_2.13-3.5.1/NOTICE
kafka_2.13-3.5.1/bin/
kafka_2.13-3.5.1/bin/connect-distributed.sh
kafka_2.13-3.5.1/bin/connect-mirror-maker.sh
kafka_2.13-3.5.1/bin/connect-standalone.sh
kafka_2.13-3.5.1/bin/kafka-acls.sh
kafka_2.13-3.5.1/bin/kafka-broker-api-versions.sh
kafka_2.13-3.5.1/bin/kafka-cluster.sh
kafka_2.13-3.5.1/bin/kafka-configs.sh
kafka_2.13-3.5.1/bin/kafka-console-consumer.sh
kafka_2.13-3.5.1/bin/kafka-console-producer.sh
kafka_2.13-3.5.1/bin/kafka-consumer-groups.sh
kafka_2.13-3.5.1/bin/kafka-consumer-perf-test.sh
kafka_2.13-3.5.1/bin/kafka-delegation-tokens.sh
kafka_2.13-3.5.1/bin/kafka-delete-records.sh
kafka_2.13-3.5.1/bin/kafka-dump-log.sh
kafka_2.13-3.5.1/bin/kafka-e2e-latency.sh
kafka_2.13-3.5.1/bin/kafka-features.sh
kafka_2.13-3.5.1/bin/kafka-get-offsets.sh
kafka_2.13-3.5.1/bin/kafka-jmx.sh
kafka_2.13-3.5.1/bin/kafka-leader-election.sh
kafka_2.13-3.5.1/bin/kafka-log-dirs.sh
kafka_2.13-3.5.1/bin/kafka-metadata-quorum.sh
kafka_2.1

In [ ]:
!./kafka_2.13-3.5.1/bin/zookeeper-server-start.sh -daemon ./kafka_2.13-3.5.1/config/zookeeper.properties
!./kafka_2.13-3.5.1/bin/kafka-server-start.sh -daemon ./kafka_2.13-3.5.1/config/server.properties
!echo "Waiting for 10 secs until kafka and zookeeper services are up and running"
!sleep 10


Waiting for 10 secs until kafka and zookeeper services are up and running


In [ ]:
!ps -ef | grep kafka

root        1569     613 12 00:59 ?        00:01:27 /usr/lib/jvm/java-11-openjdk-amd64/bin/java -cp /usr/local/lib/python3.10/dist-packages/pyspark/conf:/usr/local/lib/python3.10/dist-packages/pyspark/jars/* -Xmx1g -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED -Djdk.reflect.useDirectMethodHandle=false org.apache.spark.

In [ ]:
#!./kafka_2.13-3.5.1/bin/windows/kafka-topics.bat
!./kafka_2.13-3.5.1/bin/kafka-topics.sh --create --bootstrap-server 127.0.0.1:9092 --replication-factor 1 --partitions 1 -topic retrasos

Created topic retrasos.


In [ ]:
!./kafka_2.13-3.5.1/bin/kafka-topics.sh --describe --bootstrap-server 127.0.0.1:9092 -topic retrasos

Topic: retrasos	TopicId: ezr0QL7QTbasOA6yYASY8w	PartitionCount: 1	ReplicationFactor: 1	Configs: 
	Topic: retrasos	Partition: 0	Leader: 0	Replicas: 0	Isr: 0


In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget https://downloads.apache.org/spark/spark-3.2.4/spark-3.2.4-bin-hadoop2.7.tgz
!tar -xvf spark-3.2.4-bin-hadoop2.7.tgz
!pip install pyspark
!pip install kafka-python

--2023-09-26 01:11:19--  https://downloads.apache.org/spark/spark-3.2.4/spark-3.2.4-bin-hadoop2.7.tgz
Resolving downloads.apache.org (downloads.apache.org)... 88.99.95.219, 135.181.214.104, 2a01:4f8:10a:201a::2, ...
Connecting to downloads.apache.org (downloads.apache.org)|88.99.95.219|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 272938638 (260M) [application/x-gzip]
Saving to: ‘spark-3.2.4-bin-hadoop2.7.tgz.1’

spark-3.2.4-bin-had 100%[===================>] 260.29M  5.19MB/s    in 37s     

2023-09-26 01:11:57 (7.11 MB/s) - ‘spark-3.2.4-bin-hadoop2.7.tgz.1’ saved [272938638/272938638]

spark-3.2.4-bin-hadoop2.7/
spark-3.2.4-bin-hadoop2.7/R/
spark-3.2.4-bin-hadoop2.7/R/lib/
spark-3.2.4-bin-hadoop2.7/R/lib/sparkr.zip
spark-3.2.4-bin-hadoop2.7/R/lib/SparkR/
spark-3.2.4-bin-hadoop2.7/R/lib/SparkR/html/
spark-3.2.4-bin-hadoop2.7/R/lib/SparkR/html/00Index.html
spark-3.2.4-bin-hadoop2.7/R/lib/SparkR/html/R.css
spark-3.2.4-bin-hadoop2.7/R/lib/SparkR/INDEX
spark-3.

In [ ]:
#enlazar spark con kafka
!wget "https://repo1.maven.org/maven2/org/apache/spark/spark-streaming-kafka-0-8-assembly_2.11/2.4.8/spark-streaming-kafka-0-8-assembly_2.11-2.4.8.jar"

--2023-09-26 01:12:27--  https://repo1.maven.org/maven2/org/apache/spark/spark-streaming-kafka-0-8-assembly_2.11/2.4.8/spark-streaming-kafka-0-8-assembly_2.11-2.4.8.jar
Resolving repo1.maven.org (repo1.maven.org)... 199.232.192.209, 199.232.196.209, 2a04:4e42:4c::209, ...
Connecting to repo1.maven.org (repo1.maven.org)|199.232.192.209|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12002039 (11M) [application/java-archive]
Saving to: ‘spark-streaming-kafka-0-8-assembly_2.11-2.4.8.jar’

spark-streaming-kaf 100%[===================>]  11.45M  9.85MB/s    in 1.2s    

2023-09-26 01:12:28 (9.85 MB/s) - ‘spark-streaming-kafka-0-8-assembly_2.11-2.4.8.jar’ saved [12002039/12002039]



In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.2.4-bin-hadoop2.7"
#--packages org.apache.spark:spark-streaming-kafka-0-8_2.11:2.1.0,org.apache.spark:spark-sql-kafka-0-10_2.11:2.1.0,com.databricks:spark-avro_2.11:3.2.0 pyspark-shell
os.environ['PYSPARK_SUBMIT_ARGS'] = '--jars /content/spark-streaming-kafka-0-8-assembly_2.11-2.4.8.jar pyspark-shell'

In [ ]:
!pip install findspark

In [ ]:
import findspark
findspark.init()

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import pyspark
import sys
from pyspark import SparkContext, SparkConf
import time

In [ ]:
kafka_topic_name = "retrasos"
kafka_bootstrap_servers = 'localhost:9092'

In [ ]:
from datetime import datetime

now = datetime.now()

current_time = now.strftime("%H:%M:%S")
print("Current Time =", current_time)

Current Time = 01:13:12


In [ ]:
retrasosStreamingDF = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", kafka_topic_name) \
    .load()


In [ ]:
# Mostramos el esquema de este DataFrame
types = retrasosStreamingDF.dtypes
assert(retrasosStreamingDF.isStreaming)
assert((types[0][0] == "key")       & (types[0][1] == "binary"))
assert((types[1][0] == "value")     & (types[1][1] == "binary"))
assert((types[2][0] == "topic")     & (types[2][1] == "string"))
assert((types[3][0] == "partition") & (types[3][1] == "int"))
assert((types[4][0] == "offset")    & (types[4][1] == "bigint"))
assert((types[5][0] == "timestamp") & (types[5][1] == "timestamp"))
assert((types[6][0] == "timestampType") & (types[6][1] == "int"))

retrasosStreamingDF.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



Muestra por pantalla el esquema del DataFrame resultante de la lectura con `printSchema()`. Verás que todas estas columnas son creadas automáticamente por Spark cuando leemos de Kafka. De ellas, la que nos interesa es `value` que contiene propiamente el mensaje de Kafka, en formato datos binarios.

### Ejercicio 2

Tendremos que estructurar estos datos para poder extraer los campos. Para ello sigue los siguientes pasos, ayudándote de la plantilla que hay en la celda siguiente (descoméntala y complétala):

* **Selecciona** la columna `value` y conviértela (`.cast`) a `StringType()` utilizando `withColumn` para reemplazar la columna existente `"value"` por el objeto Column resultante de la conversión. De esta forma tendremos una columna que contendrá en cada **fila** un **fichero JSON completo**, tal como se muestra en cada una de las plantillas anteriores.
* Para extraer los dos campos de cada uno de los JSON y convertirlos en una columna llamada `parejas`, de tipo `struct` (una estructura formada por dos campos de tipo String e Integer respectivamente), utilizamos la función `from_json` de Spark, que se aplica a cada elemento (cada fila) de la columna "value" y parsea el String según un esquema que le indiquemos, devolviendo una columna de tipo `struct`.
* La columna `parejas` es de tipo `struct` por lo que puedes acceder a cada uno de sus dos campos (`dest` y `arr_delay`) con el operador `.` (punto). Utilizando `withColumn` dos veces, crea dos columnas llamadas `dest` y `arr_delay` como el resultado de acceder a `parejas.dest` y `parejas.arr_delay` respectivamente.

In [ ]:
from pyspark.sql.functions import from_json
#from pyspark.sql.functions import from_json
#from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

In [ ]:
# Definimos el esquema para el JSON
esquema = StructType([\
  StructField("dest", StringType()),\
  StructField("arr_delay", DoubleType())\
])

In [ ]:
parsedDF=retrasosStreamingDF

parsedDF=retrasosStreamingDF\
     .select("value")\
     .withColumn("value", col("value").cast(StringType()))\
     .withColumn("parejas", from_json(col("value"),esquema))\
     .withColumn("dest",col("parejas.dest"))\
     .withColumn("arr_delay",col("parejas.arr_delay"))

In [ ]:
#parsedDF=retrasosStreamingDF

#parsedDF=retrasosStreamingDF\
     #.select("value")\
     #.withColumn("value", col("value").cast(StringType()))\
     #.withColumn("parejas", from_json(col("value"),esquema))\
     #.withColumn("arr_delay",col("parejas.arr_delay"))\
     #.drop("parejas")

In [ ]:
parsedDF.printSchema()

root
 |-- value: string (nullable = true)
 |-- parejas: struct (nullable = true)
 |    |-- dest: string (nullable = true)
 |    |-- arr_delay: double (nullable = true)
 |-- dest: string (nullable = true)
 |-- arr_delay: double (nullable = true)



In [ ]:
# Realizamos la validación
tipos = parsedDF.dtypes
assert(("value", "string") in tipos)
assert(('parejas', 'struct<dest:string,arr_delay:double>') in tipos)
assert(('dest', 'string') in tipos)
assert(('arr_delay', 'double') in tipos)


Nuestro DataFrame ya contiene una columna `dest` con el nombre del aeropuerto destino y una columna de números reales `arr_delay` con el retraso. Ya podemos efectuar el mismo tipo de agregación que estamos haciendo en nuestra función `retrasoMedio`. Por tanto, invocamos a `retrasoMedio` pasando `parsedDF` como argumento.

In [ ]:
# Evalúa el siguiente código pero no lo modifiques
# Indicamos que este DataFrame se guarde en memoria cuando se va actualizando,
# y arrancamos la ejecución en Streaming con la acción start()

retrasoMedioStreamingDF = retrasoMedio(parsedDF)

consoleOutput = retrasoMedioStreamingDF\
                    .writeStream\
                    .queryName("retrasosAgg")\
                    .outputMode("complete")\
                    .format("memory")\
                    .start()


Una vez evaluada la celda anterior, abre el productor de Kafka console entrando por SSH a cualquiera de las máquinas (revisa el enunciado de la práctica para recordarlo), y copia y pega (literalmente) los siguientes 4 mensajes en formato JSON. Como ves,  tienen un campo `dest` y un campo `arr_delay`, simulando la información que estaríamos recibiendo en tiempo real de los distintos aeropuertos a medida que los vuelos van aterrizando.

Cada vez que pegues un mensaje, ejecuta la consulta `select * from retrasosAgg` a través del método `spark.sql(...)` y muestra el DataFrame `agregadosDF` devuelto por dicho método. Eso hará una consulta contra la vista temporal (volátil) `retrasosAgg` que se ha creado en el metastore de Hive gracias al `writeStream` del apartado anterior. Ejecuta la celda de `show` tantas veces como sea necesario hasta ver un resultado distinto al que has visto en la ejecución anterior, para asegurarte de que Spark ya ha leído e incorporado el nuevo dato en su cálculo de la agregación y por tanto ha actualizado el resultado.

Recuerda que el método `.sql(...)` es una transformación, y por tanto, se re-ejecuta la consulta cada vez que invocas a la acción `show()` sobre el resultado, ya que **no vamos a cachear nada**, precisamente para forzar la reevaluación de la consulta y poder ver así el contenido actualizado de dicha tabla (en memoria) de Hive cada vez que hacemos `show()`.

Se pide:
* Cada vez que envíes un mensaje y te hayas asegurado de que Spark ha incorporado el dato a su cálculo, apunta el resultado de la agregación (valor de la columna `retraso_medio`) para MAD y GRX en las variables habilitadas para ello
* No te preocupes por evaluar muchas veces una misma celda, ya que el cálculo sólo se actualizará una vez. Las siguientes veces que la evalúes seguirá mostrando el mismo resultado mientras no envíes otro nuevo mensaje en Kafka.

Los 4 mensajes que hay que introducir sucesivamente en Kafka son:

`
{"dest": "GRX", "arr_delay": 2.6}
{"dest": "MAD", "arr_delay": 5.4}
{"dest": "GRX", "arr_delay": 1.5}
{"dest": "MAD", "arr_delay": 20.0}
`

In [ ]:
from kafka import KafkaProducer
from time import sleep
from pyspark.sql import SparkSession
import json

In [ ]:
producer = KafkaProducer(bootstrap_servers='localhost:9092')

In [ ]:
# primer mensaje
mensajes= [
{"dest": "GRX", "arr_delay": 2.6}
]

In [ ]:
for mensaje in mensajes:
  mensaje_json = json.dumps(mensaje)
  producer.send('retrasos',value=mensaje_json.encode('utf-8'))

sleep(10)

In [ ]:
agregadosDF = spark.sql("select * from retrasosAgg")
agregadosDF.show()

+----+-------------+
|dest|retraso_medio|
+----+-------------+
| GRX|          2.6|
+----+-------------+



In [ ]:
columnas =agregadosDF.columns
assert(len(columnas)==2)
assert("dest" in columnas)
assert("retraso_medio" in columnas)

In [ ]:
# segundo mensaje
mensajes= [
    {"dest": "MAD", "arr_delay": 5.4}

]

In [ ]:
for mensaje in mensajes:
  mensaje_json = json.dumps(mensaje)
  producer.send('retrasos',value=mensaje_json.encode('utf-8'))

sleep(10)

In [ ]:
agregadosDF = spark.sql("select * from retrasosAgg")
agregadosDF.show()

+----+-------------+
|dest|retraso_medio|
+----+-------------+
| MAD|          5.4|
| GRX|          2.6|
+----+-------------+



In [ ]:
mensajes= [
    {"dest": "GRX", "arr_delay": 1.5}

]

In [ ]:
for mensaje in mensajes:
  mensaje_json = json.dumps(mensaje)
  producer.send('retrasos',value=mensaje_json.encode('utf-8'))

sleep(10)

In [ ]:
agregadosDF = spark.sql("select * from retrasosAgg")
agregadosDF.show()


+----+-------------+
|dest|retraso_medio|
+----+-------------+
| MAD|          5.4|
| GRX|         2.05|
+----+-------------+



In [ ]:
mensajes= [
    {"dest": "MAD", "arr_delay": 20}

]

In [ ]:
for mensaje in mensajes:
  mensaje_json = json.dumps(mensaje)
  producer.send('retrasos',value=mensaje_json.encode('utf-8'))

sleep(10)


In [ ]:
agregadosDF = spark.sql("select * from retrasosAgg")
agregadosDF.show()

+----+-------------+
|dest|retraso_medio|
+----+-------------+
| MAD|         12.7|
| GRX|         2.05|
+----+-------------+

